In [1]:
import pandas as pd
from pathlib import Path

# =========================================================
# EDITA SOLO ESTO
# =========================================================
TEAM = 7
MY_COMUNAS = [13201]   # pon aquí tus comunas reales

CENSUS_INPUT = "/Users/martingomez/Downloads/viv_hog_per_censo2024/output/tarea1_comuna_summary.csv"
ENO_INPUT    = "/Users/martingomez/Downloads/output/tarea2_eno_summary.csv"
GRD_INPUT    = "/Users/martingomez/Downloads/output/tarea2_grd_summary.csv"
# =========================================================


# =========================================================
# ESQUEMAS OFICIALES
# =========================================================
CENSUS_COLS = [
    ("codigo_comuna", "int"),
    ("nombre_comuna", "str"),
    ("pop_total", "int"),
    ("pop_chilean", "int"),
    ("pop_foreign", "int"),
    ("pct_foreign", "float"),
    ("median_age_chilean", "float"),
    ("median_age_foreign", "float"),
    ("mean_schooling_chilean", "float"),
    ("mean_schooling_foreign", "float"),
    ("emp_rate_chilean", "float"),
    ("emp_rate_foreign", "float"),
    ("dependency_ratio", "float"),
]

ENO_COLS = [
    ("codigo_comuna", "int"),
    ("nombre_comuna", "str"),
    ("eno_total", "int"),
    ("eno_chilean", "int"),
    ("eno_foreign", "int"),
    ("eno_desconocido", "int"),
    ("eno_top3_diseases", "str"),
    ("eno_rate_per_10k", "float"),
]

GRD_COLS = [
    ("codigo_comuna", "int"),
    ("nombre_comuna", "str"),
    ("grd_total", "int"),
    ("grd_chilean", "int"),
    ("grd_foreign", "int"),
    ("grd_pct_foreign", "float"),
    ("grd_mean_los", "float"),
    ("grd_mean_los_chilean", "float"),
    ("grd_mean_los_foreign", "float"),
    ("grd_mean_severity", "float"),
    ("grd_mortality_rate", "float"),
    ("grd_top3_chapters", "str"),
    ("grd_rate_per_10k", "float"),
]

CENSUS_EXPECTED = [c for c, _ in CENSUS_COLS]
ENO_EXPECTED    = [c for c, _ in ENO_COLS]
GRD_EXPECTED    = [c for c, _ in GRD_COLS]


def ensure_columns(df, expected_cols, df_name):
    missing = [c for c in expected_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{df_name}: faltan columnas {missing}")
    return df[expected_cols].copy()

def maybe_convert_fraction_to_percent(series):
    s = pd.to_numeric(series, errors="coerce")
    non_null = s.dropna()
    if len(non_null) and ((non_null >= 0) & (non_null <= 1)).all():
        return s * 100
    return s

def clean_top3_string(series):
    return (
        series.astype(str)
        .str.replace(", ", " | ", regex=False)
        .str.replace(",", " | ", regex=False)
        .str.strip()
    )

def _check_dtype(series, kind, col):
    if kind == "int":
        assert pd.api.types.is_integer_dtype(series), f"{col}: expected int, got {series.dtype}"
    elif kind == "float":
        assert pd.api.types.is_float_dtype(series) or pd.api.types.is_integer_dtype(series), \
            f"{col}: expected float, got {series.dtype}"
    elif kind == "str":
        assert series.dtype == object, f"{col}: expected string, got {series.dtype}"

def validate(path, schema, my_comunas):
    path = Path(path)
    assert path.exists(), f"{path} does not exist"
    df = pd.read_csv(path)

    expected_cols = [c for c, _ in schema]
    assert list(df.columns) == expected_cols, (
        f"{path}: columns must be exactly {expected_cols}, got {list(df.columns)}"
    )

    for col, kind in schema:
        _check_dtype(df[col], kind, col)

    assert len(df) == len(my_comunas), (
        f"{path}: expected {len(my_comunas)} rows, got {len(df)}"
    )
    assert set(df["codigo_comuna"]) == set(my_comunas), (
        f"{path}: codigo_comuna must be exactly {set(my_comunas)}, got {set(df['codigo_comuna'])}"
    )
    assert df["codigo_comuna"].is_unique, f"{path}: codigo_comuna must be unique"

    print(f"OK  {path.name:30s}  rows={len(df)}  cols={len(df.columns)}")


# CARGAR RESÚMENES
census = pd.read_csv(CENSUS_INPUT)
eno    = pd.read_csv(ENO_INPUT)
grd    = pd.read_csv(GRD_INPUT)

# AJUSTAR CENSUS
census = ensure_columns(census, CENSUS_EXPECTED, "census")
for col in ["pct_foreign", "emp_rate_chilean", "emp_rate_foreign", "dependency_ratio"]:
    census[col] = maybe_convert_fraction_to_percent(census[col])

for col in ["codigo_comuna", "pop_total", "pop_chilean", "pop_foreign"]:
    census[col] = pd.to_numeric(census[col], errors="raise").astype("int64")

for col in [
    "pct_foreign", "median_age_chilean", "median_age_foreign",
    "mean_schooling_chilean", "mean_schooling_foreign",
    "emp_rate_chilean", "emp_rate_foreign", "dependency_ratio"
]:
    census[col] = pd.to_numeric(census[col], errors="coerce").astype("float64")

census["nombre_comuna"] = census["nombre_comuna"].astype(str)

# AJUSTAR ENO
eno = ensure_columns(eno, ENO_EXPECTED, "eno")
eno["eno_top3_diseases"] = clean_top3_string(eno["eno_top3_diseases"])

for col in ["codigo_comuna", "eno_total", "eno_chilean", "eno_foreign", "eno_desconocido"]:
    eno[col] = pd.to_numeric(eno[col], errors="raise").astype("int64")

eno["eno_rate_per_10k"] = pd.to_numeric(eno["eno_rate_per_10k"], errors="coerce").astype("float64")
eno["nombre_comuna"] = eno["nombre_comuna"].astype(str)
eno["eno_top3_diseases"] = eno["eno_top3_diseases"].astype(str)

# AJUSTAR GRD
grd = ensure_columns(grd, GRD_EXPECTED, "grd")
for col in ["grd_pct_foreign", "grd_mortality_rate"]:
    grd[col] = maybe_convert_fraction_to_percent(grd[col])

grd["grd_top3_chapters"] = clean_top3_string(grd["grd_top3_chapters"])

for col in ["codigo_comuna", "grd_total", "grd_chilean", "grd_foreign"]:
    grd[col] = pd.to_numeric(grd[col], errors="raise").astype("int64")

for col in [
    "grd_pct_foreign", "grd_mean_los", "grd_mean_los_chilean",
    "grd_mean_los_foreign", "grd_mean_severity",
    "grd_mortality_rate", "grd_rate_per_10k"
]:
    grd[col] = pd.to_numeric(grd[col], errors="coerce").astype("float64")

grd["nombre_comuna"] = grd["nombre_comuna"].astype(str)
grd["grd_top3_chapters"] = grd["grd_top3_chapters"].astype(str)

# FILTRAR TUS COMUNAS
census = census[census["codigo_comuna"].isin(MY_COMUNAS)].sort_values("codigo_comuna").reset_index(drop=True)
eno    = eno[eno["codigo_comuna"].isin(MY_COMUNAS)].sort_values("codigo_comuna").reset_index(drop=True)
grd    = grd[grd["codigo_comuna"].isin(MY_COMUNAS)].sort_values("codigo_comuna").reset_index(drop=True)

# EXPORTAR
team_tag = f"team{TEAM:02d}"
census_out = f"census_{team_tag}.csv"
eno_out    = f"eno_{team_tag}.csv"
grd_out    = f"grd_{team_tag}.csv"

census.to_csv(census_out, index=False, encoding="utf-8", na_rep="")
eno.to_csv(eno_out, index=False, encoding="utf-8", na_rep="")
grd.to_csv(grd_out, index=False, encoding="utf-8", na_rep="")

print("Archivos exportados:")
print(census_out)
print(eno_out)
print(grd_out)

# VALIDAR
validate(census_out, CENSUS_COLS, MY_COMUNAS)
validate(eno_out, ENO_COLS, MY_COMUNAS)
validate(grd_out, GRD_COLS, MY_COMUNAS)
print("All three files OK -- ready to upload.")

Archivos exportados:
census_team07.csv
eno_team07.csv
grd_team07.csv
OK  census_team07.csv               rows=1  cols=13
OK  eno_team07.csv                  rows=1  cols=8
OK  grd_team07.csv                  rows=1  cols=13
All three files OK -- ready to upload.
